# Run and plot a `LAPDSim1D` simulation

This notebook:
1. Builds a `LAPDSim1D` from `default_config()`, applies your **parameter/flag overrides**, runs it, and saves an HDF5 result.
2. Renders the app-style contour and summary plots inline.
3. Shows main-discharge time-slice profiles at **15 ms and 19 ms** only.

All z-axis plots include the vertical dashed **port markers** (ports 20, 29, 40) used by `bapsf_app`.

Run the notebook from `cablp/scripts/` (it imports the CLI helper `plot_sim1d_run.py` from that directory).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from cablp.solvers._sim1d import (
    LAPDSim1D,
    ProgressPrinter1D,
    default_config,
    load_result_hdf5,
    summarize_result,
)

# The CLI plotting helpers live next to this notebook in cablp/scripts/.
# Importing the module sets the Agg backend, so re-assert the inline backend after.
sys.path.insert(0, str(Path.cwd()))
import plot_sim1d_run as psr
%matplotlib inline

## Configuration

Edit `param_overrides` / `flag_overrides` to override the defaults. Anything left commented uses the value from `default_config()`. Call `default_config()` in a scratch cell to see every available key.

The run controls below map to `sim.start_simulation(...)`; leave them `None` to use the config defaults.

In [ ]:
# --- Parameter overrides (input_dict keys) ---
Ts = 273.15 + 1700

param_overrides = {
    # "gas_type": "He",
    # "nx": 62,
    "V_bank": 180.0,
    "T_s": Ts,
    "S_gp": 4000,
    "S_gp_decay_target": 1500,
    "tau_gp_pulse_duration": 1e-3,
    "tau_gp_decay_duration": 5e-3,
    "b_ion_neutral_drag": 0.5,
    "b_Qei": 0.5,
    "b_Qen": 0.5,
    "b_Qcx": 0.5,
    "Rp": 15.0,
    "R_cath": 15.0,
    "R_comp": 0.010,
    # Ion-neutral thermal equilibration (enabled via the flag below) relaxes Ti
    # toward Tn_fit, NOT toward Tn_K. Tn_fit defaults to 0.1 eV = 1160 K while
    # Tn_K is 300 K = 0.026 eV, and Tn_fit is documented as the neutral
    # temperature used by the *reaction-rate fits*. Note 0.1 eV is also exactly
    # Ti_floor, so as written the equilibration drives Ti onto the floor rather
    # than to the gas temperature. Set Tn_fit here if a different target is
    # intended -- but a target below Ti_floor will fight the floor.
    # "Tn_fit": 0.1,
    #
    # --- Time integration (operator-split path only) ---
    # The three keys below are what make the split step second-order, and they
    # are three SEPARATE first-order error terms: leaving any one at its default
    # caps the whole step at 1st order no matter what the other two say. Measured
    # order, scripts/verify_sim1d_order.py at 62 cells:
    #
    #   picard  splitting   backward_euler  crank_nicolson  tr_bdf2
    #     0       lie            0.97            1.01         1.02
    #     4       lie            0.97            1.01         1.02
    #     0       strang         0.98            1.04         0.98
    #     4       strang         0.99            1.99         2.00
    #
    # backward_euler staying at ~1.0 everywhere is the negative control: it
    # cannot be 2nd order at any dt. Full package costs ~1.3-1.45x wall clock;
    # the extra tridiagonal solves are cheap next to the reaction-rate
    # evaluations, which run the same number of times either way.
    #
    # NB a production discharge will NOT show 2nd order: the floors bind on ~42%
    # of cell-visits and phase transitions are threshold-triggered, so the step
    # degrades to 1st order wherever those engage. See NUMERICS.md.
    #
    # Substep discretization:
    #   "backward_euler"  theta=1    over-diffusive but unconditionally monotone,
    #                                so it cannot undershoot the floors
    #   "shifted"         theta=0.6  ~1/5 of backward Euler's error constant;
    #                                damps stiff ringing ~2/3 per step
    #   "crank_nicolson"  theta=0.5  2nd-order, but rings at undamped amplitude
    #                                on stiff modes (not L-stable)
    #   "tr_bdf2"                    2nd-order AND L-stable (trapezoidal stage
    #                                then BDF2): ~60x less ringing than CN and
    #                                half its error, for 2 solves not 1
    # Audited on this config: crank_nicolson clips the Te floor once in ~20k
    # solves (at plasma launch, ~1e-10 of the column thermal energy); tr_bdf2
    # and backward_euler never clip.
    "implicit_heat_scheme": "tr_bdf2",
    # Operator composition. "lie" = A(dt) then B(dt), 1st order. "strang" =
    # B(dt/2), A(dt), B(dt/2), cancelling the leading dt*[A,B] commutator term.
    # Costs one extra heat substep, not one extra explicit step.
    "operator_splitting": "strang",
    # Conductivity evaluation. 0 freezes Braginskii kappa ~ T^2.5 at the step
    # start, which is 1st order however good the substep scheme is. 1 iteration
    # already reaches 2nd order; 2 buys fixed-point margin at large dt for
    # ~0.15x more wall clock. Not worth enabling for backward_euler/shifted --
    # it cannot help their order and slightly worsens their error constant.
    "heat_picard_iterations": 2,
    "heat_picard_tol": 1e-10,
    #
    # "sigma_in_cm2": 5.0e-15,        # ion-neutral cross section
    # "b_ion_neutral_drag": 1.0,      # ion-neutral drag scale
    # "end_surface_area_scale": 1.8,  # end-wall neutralization area
}

# --- Flag overrides (input_flags keys) ---
flag_overrides = {
    # "implicit_heat_conduction": True,
    # "ion_neutral_drag": True,
    "ion_neutral_drag_cx_only": False,
    # Elastic ion-neutral thermal equilibration: relaxes Ti toward Tn_fit at the
    # elastic collision rate, the elastic companion to the CX cooling Q_cx. It is
    # scaled by b_ion_neutral_drag (0.5 above) and vanishes entirely if
    # ion_neutral_drag_cx_only is True, since then there is no elastic fraction.
    # See the Tn_fit note in param_overrides for what it equilibrates *to*.
    "ion_neutral_thermalization": True,
}

# --- Run controls (None => config default) ---
t_end = None            # [s] final time
dt = None               # [s] fixed step; None => adaptive
operator_split = None   # None => use implicit_heat_conduction flag
max_steps = None        # accepted-step cap; 0 => unlimited

output_path = "sim1d_run.h5"

## Run the simulation

In [ ]:
params, flags = default_config()
params.update(param_overrides)
flags.update(flag_overrides)

sim = LAPDSim1D(params, flags)
sim.start_simulation(
    t_end=t_end,
    dt=dt,
    operator_split=operator_split,
    max_steps=max_steps,
    progress_tracker=ProgressPrinter1D(),
)

out = Path(output_path)
out.parent.mkdir(parents=True, exist_ok=True)
sim.save_result(out, sim.get_results(), params=params, flags=flags)

# Reload from HDF5 so `result` carries params/flags and sim3-compatible aliases.
# The in-memory get_results() result has no .params, which otherwise shows up as
# NaN (V_bank, S_gp, ...) in the plot titles. This matches how the CLI plot
# script consumes results.
result = load_result_hdf5(out)

s = summarize_result(result)
print(f"steps={result.steps}, final_time={result.final_time:.4e} s, saves={len(result.time)}, output={out}")
print(
    f"finite={s.finite}, "
    f"n=[{s.n_min:.3e}, {s.n_max:.3e}] cm^-3, "
    f"Te=[{s.Te_min:.3e}, {s.Te_max:.3e}] eV, "
    f"Ti=[{s.Ti_min:.3e}, {s.Ti_max:.3e}] eV"
)

## Plots

Port markers are the vertical gray lines used by `bapsf_app`, at the LAPD probe ports (sim convention, z=0 at the source end):

| port | z [cm] | style |
|------|--------|-------|
| 11   | 470    | dotted |
| 20   | 758    | dashed |
| 29   | 1045   | dashed |
| 40   | 1397   | dashed |
| 50   | 1717   | dotted |

Edit `PORT_Z_SIM_CM` (dashed) / `PORT_Z_DOTTED_CM` (dotted) if your geometry differs.

In [ ]:
# Port probe positions [cm], sim convention (z=0 at source end).
# LAPD ports are ~31.95 cm apart; z ~= 31.95*port + 119 cm (matches bapsf_app's
# hardcoded 20/29/40 markers).
PORT_Z_SIM_CM = {20: 758.0, 29: 1045.0, 40: 1397.0}   # dashed markers
PORT_Z_DOTTED_CM = {11: 470.0, 50: 1717.0}            # dotted markers
ALL_PORT_Z_CM = {**PORT_Z_DOTTED_CM, **PORT_Z_SIM_CM}


def add_port_lines(ax):
    """Draw the bapsf_app vertical port markers on a z-axis plot."""
    for z in PORT_Z_SIM_CM.values():
        ax.axvline(z, color="gray", lw=0.8, ls="--", alpha=0.7)
    for z in PORT_Z_DOTTED_CM.values():
        ax.axvline(z, color="gray", lw=0.8, ls=":", alpha=0.7)


# Reproduce the CLI time axis (t=0 at main discharge, milliseconds).
time_origin = psr._time_origin(result, "main_discharge")
time_scale, time_label = psr._time_unit("ms")
shifted_s = np.asarray(result.time, dtype=float) - time_origin
t_plot = shifted_s * time_scale
t_slice_ms = shifted_s * 1.0e3
z_cm = np.asarray(result.z_cm, dtype=float)
phase_events = psr._shifted_phase_events(result, time_origin, time_scale)

In [ ]:
figures = {
    "Summary": psr._plot_summary(result, t_plot, time_label, phase_events),
    "Densities": psr._plot_densities(result, z_cm, t_plot, time_label, phase_events),
    "Temperatures": psr._plot_temperatures(result, z_cm, t_plot, time_label, phase_events),
    "Velocity": psr._plot_velocity(result, z_cm, t_plot, time_label, phase_events),
    "Energy terms": psr._plot_energy_terms(result, t_plot, time_label, phase_events),
    "Cathode": psr._plot_cathode(result, t_plot, time_label, phase_events),
    "Phase": psr._plot_phase(result, t_plot, time_label, phase_events),
}

for name, fig in figures.items():
    if fig is None:
        continue
    # Add port markers only to panels whose x-axis is z [cm] (skips time-axis and colorbar axes).
    for ax in fig.axes:
        if ax.get_xlabel().startswith("z"):
            add_port_lines(ax)
    display(fig)
    plt.close(fig)

## Time-slice profiles at 15 ms and 19 ms

Main-discharge-relative times. Each figure snaps to the nearest saved timestep; the title reports the actual time used. Make sure the run reaches ~19 ms of main discharge, or edit `SLICE_TIMES_MS`.

In [ ]:
SLICE_TIMES_MS = (15.0, 19.0)

for slice_ms in SLICE_TIMES_MS:
    fig = psr._plot_time_slice_summary(
        result=result,
        z_cm=z_cm,
        t_ms=t_slice_ms,
        slice_time_ms=slice_ms,
    )
    for ax in fig.axes:  # all four panels share a z [cm] x-axis
        add_port_lines(ax)
    display(fig)
    plt.close(fig)

## Port time series

`ne(t)` and `Te(t)` at the five port positions (nearest cell to each entry in `ALL_PORT_Z_CM`: dashed ports 20/29/40 and dotted ports 11/50). Vertical dashed lines mark the main-discharge/afterglow phase transitions.

In [ ]:
# ne(t) and Te(t) at the port positions.
ne = np.asarray(result.n, dtype=float)
Te = np.asarray(result.Te, dtype=float)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), constrained_layout=True)
for port in sorted(ALL_PORT_Z_CM):
    z = ALL_PORT_Z_CM[port]
    idx = int(np.argmin(np.abs(z_cm - z)))  # nearest cell to the port
    label = f"port {port} (z={z_cm[idx]:.0f} cm)"
    axes[0].plot(t_plot, ne[:, idx], label=label)
    axes[1].plot(t_plot, Te[:, idx], label=label)

axes[0].set_ylabel(r"$n_e$ [cm$^{-3}$]")
axes[0].set_yscale("linear")
axes[0].set_title("Electron density at ports")
axes[1].set_ylabel(r"$T_e$ [eV]")
axes[1].set_title("Electron temperature at ports")
axes[0].set_ylim(0,2e13)
axes[1].set_ylim(0,12)
for ax in axes:
    ax.set_xlabel(time_label)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    psr._add_phase_lines(ax, phase_events)  # main_discharge / afterglow markers

fig.suptitle(f"Port time series\n{psr._plot_title(result)}", fontsize=11)
display(fig)
plt.close(fig)

## Heat-term time slices

A curated set of the most important energy source terms, each as a line of its **absolute value** in W/cm² vs z (converted from the stored W/cm³ via `length_cm`), with **▲ markers where the term heats** and **▼ where it cools** that species. Terms are grouped in `HEAT_TERM_GROUPS` — e.g. **net beam** = deposition + birth + cost, **recombination** = radiative + 3-body — and surface/end-loss and pure transport terms are dropped. Same slice times as above (`SLICE_TIMES_MS`); groups that are ~0 for a species are omitted. Edit `HEAT_TERM_GROUPS` to add/remove terms. Styled after `bapsf_app`'s heat-term slices.

In [ ]:
# Heat-term time slices (bapsf-app style): |grouped source term| in W/cm^2 vs z,
# with up markers where the term heats and down markers where it cools.
length_cm = np.asarray(result.length_cm, dtype=float)  # dz = V_cell / A_plasma
HEAT_FLOOR_W_CM2 = 1e-9  # ignore terms below this; also the log-axis floor

# Curated groups of the most important terms; each label sums its signed
# component term keys (stored W/cm^3). Components that are ~0 for a species drop
# out automatically, so the same set works for both panels. Surface/end-loss and
# pure transport (advective/front flux, pressure work) terms are omitted here --
# see the volume-integrated budget below for those.
HEAT_TERM_GROUPS = {
    "net beam": ["beam_power_deposition", "beam_ionization_birth", "beam_ionization_cost"],
    "e-i exchange": ["ei_exchange"],
    "electron-ion cooling": ["electron_ion_cooling"],
    "electron-neutral cooling": ["electron_neutral_cooling"],
    "ion charge exchange (Qcx)": ["ion_charge_exchange"],
    "ion-neutral thermalization": ["ion_neutral_thermalization"],
    "heat conduction": ["heat_conduction"],
    "recombination": ["recombination_rad_loss", "recombination_3b_loss"],
}


def plot_signed_abs(ax, z, y, label, color):
    """abs(y) line; up-triangle where y>0 (heating), down-triangle where y<0 (cooling)."""
    yabs = np.abs(y)
    ax.plot(z, yabs, color=color, lw=1.2, alpha=0.7, zorder=2)
    pos = y >= 0
    ax.scatter(z[pos], yabs[pos], color=color, marker="+", s=24, zorder=3, label=label)
    ax.scatter(z[~pos], yabs[~pos], color=color, marker="_", s=24, zorder=3)


def plot_heat_terms(ax, terms_dict, idx, title):
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    ci = 0
    for label, components in HEAT_TERM_GROUPS.items():
        q = np.zeros_like(length_cm)  # net signed W/cm^2 for the group
        for name in components:
            if name in terms_dict:
                q = q + np.asarray(terms_dict[name], dtype=float)[idx, :] * length_cm
        if not np.any(np.abs(q) > HEAT_FLOOR_W_CM2):
            continue  # group inactive for this species
        plot_signed_abs(ax, z_cm, q, label, colors[ci % len(colors)])
        ci += 1
    add_port_lines(ax)
    ax.set_yscale("log")
    ax.set_ylim(bottom=HEAT_FLOOR_W_CM2)
    ax.set_xlabel("z [cm]")
    ax.set_ylabel(r"$|q|$ [W cm$^{-2}$]")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best", ncol=2)


for slice_ms in SLICE_TIMES_MS:
    idx = int(np.argmin(np.abs(t_slice_ms - slice_ms)))
    actual_ms = float(t_slice_ms[idx])
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    plot_heat_terms(axes[0], result.electron_energy_terms_W_cm3, idx, "Electron heat terms")
    plot_heat_terms(axes[1], result.ion_energy_terms_W_cm3, idx, "Ion heat terms")
    fig.suptitle(
        f"Heat-term slice at {slice_ms:.1f} ms (nearest saved {actual_ms:.3f} ms)"
        "   ▲ heating / ▼ cooling\n" + psr._plot_title(result),
        fontsize=11,
    )
    display(fig)
    plt.close(fig)

## Volume-integrated energy budget

Every energy source term integrated over the plasma volume, in **watts vs time** — the same terms as the slice plots above, but summed over the whole column so the totals are directly comparable and the discharge's power balance is visible in one place. Unlike the slice plots, this **includes surface/end loss** and the transport terms.

Signs are physical: **positive heats** that species, **negative cools** it. The y-axis is symlog, so terms spanning many orders are all visible while the sign is preserved; the shaded band is the linear region near zero. Terms whose peak never exceeds `BUDGET_LINTHRESH_W` are dropped from the legend as inactive for that species.

**The net gets its own panel.** It is a near-cancellation of terms that individually reach ~1e5 W, so on a shared axis it is unreadable — the bottom panel gives it its own scale, with the electron net, the ion net, and their total.

That panel also carries a **completeness check**: the dotted green line is the finite-difference `d(thermal energy)/dt` from the saved trajectory, computed independently of the terms. It should track the total net. Where it doesn't, the gap is energy the *floors* injected — which no term reports — plus time-discretization error between saved samples. A persistent divergence would mean a source term is missing from `BUDGET_GROUPS`.

**Conservation check.** `heat_conduction`, `plasma_advective_flux` and `plasma_front_flux` are flux *divergences* over a closed domain: they only move energy between cells, so their volume integral is analytically zero. They are printed below rather than plotted — a line at ~1e-9 W tells you nothing on a plot whose other terms are ~1e5 W — and how close they land to zero, relative to the beam power, checks the conservative flux implementation. They are still counted in the net. Pressure work is *not* one of these: it is a genuine source and stays in the plot.

In [ ]:
# Volume-integrated energy budget [W] vs time.
# Terms are stored as W/cm^3 per cell; multiplying by the plasma cell volume and
# summing over z gives the total power each term puts into (or takes out of) the
# column. Sign is physical: positive heats that species.
Vp = np.asarray(result.plasma_volume_cm3, dtype=float)

BUDGET_LINTHRESH_W = 1.0  # symlog linear region [W]; also the "inactive" cutoff
NET_LINTHRESH_W = 1.0     # symlog linear region for the net panel [W]

# Grouped like HEAT_TERM_GROUPS, but adding what the slice plots omit: the
# surface/end loss and the transport terms.
BUDGET_GROUPS = {
    "beam power (incl. ohmic)": ["beam_power_deposition"],
    "beam ionization": ["beam_ionization_birth", "beam_ionization_cost"],
    "bulk ionization": ["ionization_birth", "ionization_energy_cost"],
    "e-i exchange": ["ei_exchange"],
    "electron-ion cooling": ["electron_ion_cooling"],
    "electron-neutral cooling": ["electron_neutral_cooling"],
    "ion charge exchange (Qcx)": ["ion_charge_exchange"],
    "ion-neutral thermalization": ["ion_neutral_thermalization"],
    "ion-neutral friction": ["ion_neutral_frictional_heating"],
    "recombination": ["recombination_rad_loss", "recombination_3b_loss"],
    "surface/end loss": ["surface_loss", "cathode_surface_loss"],
    "pressure work": ["pressure_work"],
}

# Flux divergences over a closed domain: these only move energy between cells, so
# their volume integral must vanish. Reported as a conservation check below
# rather than plotted, since a line at ~1e-9 W is not informative on a plot whose
# other terms are ~1e5 W. Pressure work is NOT one of these -- it is a genuine
# source and stays in BUDGET_GROUPS above. They are still counted in the net.
CONSERVATION_CHECKS = {
    "heat conduction": ["heat_conduction"],
    "advective flux": ["plasma_advective_flux"],
    "front flux": ["plasma_front_flux"],
}


def integrate_W(terms_dict, names):
    """Volume-integrate the named W/cm^3 terms over the plasma -> W vs time."""
    total = np.zeros(len(result.time), dtype=float)
    for name in names:
        if name in terms_dict:
            total = total + np.sum(
                np.asarray(terms_dict[name], dtype=float) * Vp[None, :], axis=1
            )
    return total


def plot_budget(ax, terms_dict, title):
    """Draw the individual terms; return the net for the separate net panel."""
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    net = np.zeros(len(result.time), dtype=float)
    ci = 0
    for label, components in {**BUDGET_GROUPS, **CONSERVATION_CHECKS}.items():
        p_W = integrate_W(terms_dict, components)
        net = net + p_W  # the net counts the flux terms too, even though they are ~0
        if label in CONSERVATION_CHECKS or np.max(np.abs(p_W)) < BUDGET_LINTHRESH_W:
            continue  # inactive for this species; keeps the legend readable
        ax.plot(t_plot, p_W, color=colors[ci % len(colors)], lw=1.3, label=label)
        ci += 1
    ax.axhline(0.0, color="gray", lw=0.8, alpha=0.6)
    ax.axhspan(-BUDGET_LINTHRESH_W, BUDGET_LINTHRESH_W, color="gray", alpha=0.12, lw=0)
    ax.set_yscale("symlog", linthresh=BUDGET_LINTHRESH_W)
    ax.set_ylabel("power [W]")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best", ncol=2)
    psr._add_phase_lines(ax, phase_events)
    return net


def plot_net(ax, net_e, net_i):
    """Net power per species, on its own axis.

    The net is a near-cancellation of terms that individually reach ~1e5 W, so
    sharing an axis with them makes it unreadable. Here it gets its own scale.
    """
    ax.plot(t_plot, net_e, color="tab:blue", lw=1.4, label="electron net")
    ax.plot(t_plot, net_i, color="tab:red", lw=1.4, label="ion net")
    ax.plot(t_plot, net_e + net_i, color="k", lw=1.8, ls="--", label="total net")

    # Independent check on whether the term list is complete: the finite-difference
    # rate of change of the actual thermal energy should track the total net.
    # Where it does not, the gap is energy the floors injected -- which no term
    # reports -- plus the time-discretization error between saved samples.
    E_tot_erg = np.sum(
        (np.asarray(result.Ee, dtype=float) + np.asarray(result.Ei, dtype=float))
        * Vp[None, :],
        axis=1,
    )
    dEdt_W = np.gradient(E_tot_erg, np.asarray(result.time, dtype=float)) * 1.0e-7
    ax.plot(t_plot, dEdt_W, color="tab:green", lw=1.0, ls=":", alpha=0.9,
            label="d(thermal)/dt  [finite diff]")

    ax.axhline(0.0, color="gray", lw=0.8, alpha=0.6)
    ax.axhspan(-NET_LINTHRESH_W, NET_LINTHRESH_W, color="gray", alpha=0.12, lw=0)
    ax.set_yscale("symlog", linthresh=NET_LINTHRESH_W)
    ax.set_xlabel(time_label)
    ax.set_ylabel("power [W]")
    ax.set_title("Net power  (+ heats / - cools)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best", ncol=2)
    psr._add_phase_lines(ax, phase_events)


fig, axes = plt.subplots(3, 1, figsize=(11, 13), constrained_layout=True, sharex=True)
net_e = plot_budget(axes[0], result.electron_energy_terms_W_cm3,
                    "Electron energy budget (volume-integrated)")
net_i = plot_budget(axes[1], result.ion_energy_terms_W_cm3,
                    "Ion energy budget (volume-integrated)")
plot_net(axes[2], net_e, net_i)
fig.suptitle(
    "Volume-integrated energy budget   (+ heats / - cools)\n" + psr._plot_title(result),
    fontsize=11,
)
display(fig)
plt.close(fig)

# Peak magnitude of each term: a quick numeric read of who dominates.
print(f"{'term':32} {'|peak| electron [W]':>20} {'|peak| ion [W]':>18}")
for label, components in BUDGET_GROUPS.items():
    pe = np.max(np.abs(integrate_W(result.electron_energy_terms_W_cm3, components)))
    pi = np.max(np.abs(integrate_W(result.ion_energy_terms_W_cm3, components)))
    if max(pe, pi) < BUDGET_LINTHRESH_W:
        continue
    print(f"  {label:30} {pe:20.4g} {pi:18.4g}")
print(f"  {'NET':30} {np.max(np.abs(net_e)):20.4g} {np.max(np.abs(net_i)):18.4g}")

# Conservation check: each of these is a flux divergence over a closed domain,
# so its volume integral is zero analytically. How close it lands to zero,
# relative to the beam power driving the discharge, is a check on the
# conservative flux implementation -- not a physics result.
beam_scale = max(
    np.max(np.abs(integrate_W(result.electron_energy_terms_W_cm3, ["beam_power_deposition"]))),
    1.0,
)
print(f"\nconservation checks (volume integral of a flux divergence must vanish;"
      f" beam power = {beam_scale:.3g} W):")
for label, components in CONSERVATION_CHECKS.items():
    pe = np.max(np.abs(integrate_W(result.electron_energy_terms_W_cm3, components)))
    pi = np.max(np.abs(integrate_W(result.ion_energy_terms_W_cm3, components)))
    print(f"  {label:18} |peak| e {pe:10.3e} W   i {pi:10.3e} W"
          f"   -> {max(pe, pi) / beam_scale:.1e} of beam power")